# Laboratorio de Procesamiento de Lenguaje Natural

Laboratorio académico modular compatible con Google Colab y Jupyter. Ejecuta las celdas en orden. Activa una GPU de Colab en Entorno de ejecución > Cambiar tipo de entorno si deseas acelerar los modelos. Las celdas que usan Hugging Face o spaCy requieren conexión a internet durante la primera ejecución.

In [ ]:
# %% Instalación: instala las dependencias necesarias en Google Colab o Jupyter.
# Este laboratorio no usa visión artificial; quitamos torchvision para evitar conflictos con torch.
%pip uninstall -y -q torchvision
%pip install -q -U nltk pandas spacy scikit-learn transformers sentence-transformers
!python -m spacy download es_core_news_sm

In [ ]:
# %% Configuración: importa librerías, descarga recursos, selecciona GPU/CPU y define los datos compartidos.
import re
from collections import Counter

import nltk
import pandas as pd
import spacy
import torch
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

nltk.download('stopwords', quiet=True)
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    print('El modelo de spaCy no está instalado. Descargándolo...')
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

DEVICE = 0 if torch.cuda.is_available() else -1
print('Dispositivo Transformers:', 'GPU' if DEVICE == 0 else 'CPU')

corpus_ejemplo = [
    'La inteligencia artificial transforma el análisis de datos en tiempo real.',
    'El procesamiento del lenguaje natural permite comprender el texto humano.',
    'Los modelos de lenguaje generan resúmenes y traducen textos con precisión.'
]
texto_base = corpus_ejemplo[0]

def mostrar(titulo, valor):
    print(f'\n{titulo}')
    print(valor)

## 1. Normalización de texto

La normalización reduce variaciones superficiales del texto para facilitar su análisis. En este ejemplo se convierten los caracteres a minúsculas, se eliminan signos mediante una expresión regular y se compactan los espacios. El resultado conserva las palabras principales, pero deja una representación homogénea para las técnicas posteriores.

In [ ]:
# %% 1. Normalización: convierte a minúsculas, elimina signos con regex y normaliza espacios.
def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', ' ', texto, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', texto).strip()

texto_normalizado = normalizar(texto_base)
mostrar('1. Texto normalizado:', texto_normalizado)

## 2. Stemming

El stemming aplica reglas heurísticas para recortar palabras hasta una raíz aproximada. Es útil para agrupar variantes como verbos o sustantivos relacionados, aunque la raíz resultante no siempre es una palabra válida del español. Por eso debe interpretarse como una representación para búsqueda o conteo, no como una lematización lingüística.

In [ ]:
# %% 2. Stemming: reduce palabras a una raíz aproximada con SnowballStemmer en español.
stemmer = SnowballStemmer('spanish')
palabras = texto_normalizado.split()
stems = [stemmer.stem(palabra) for palabra in palabras]
mostrar('2. Stems:', list(zip(palabras, stems)))

## 3. Stopwords

Las stopwords son palabras muy frecuentes que suelen aportar poca información temática, como artículos, preposiciones y conjunciones. Al filtrarlas se reduce el ruido léxico y se destacan términos con mayor contenido. El notebook informa si utilizó el listado de NLTK o el respaldo local cuando Colab bloquea la descarga.

In [ ]:
# %% 3. Stopwords: filtra palabras funcionales con NLTK en español y usa un respaldo si Colab bloquea la descarga.
try:
    stop_words_es = set(stopwords.words('spanish'))
    fuente_stopwords = 'NLTK'
except LookupError:
    stop_words_es = {
        'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con',
        'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde', 'el', 'ella',
        'ellas', 'ellos', 'en', 'entre', 'era', 'es', 'esa', 'esas', 'ese', 'eso',
        'esos', 'esta', 'estas', 'este', 'esto', 'estos', 'fue', 'ha', 'hacia',
        'hasta', 'la', 'las', 'le', 'les', 'lo', 'los', 'más', 'me', 'mi', 'mis',
        'muy', 'no', 'nos', 'o', 'para', 'pero', 'por', 'que', 'se', 'sin', 'sobre',
        'su', 'sus', 'también', 'te', 'tiene', 'tu', 'tus', 'un', 'una', 'unas',
        'uno', 'unos', 'y', 'ya'
    }
    fuente_stopwords = 'respaldo local'

palabras_sin_stopwords = [p for p in palabras if p not in stop_words_es]
mostrar(f'3. Palabras sin stopwords ({fuente_stopwords}):', palabras_sin_stopwords)

## Corpus tabular y trazabilidad de las transformaciones

Para documentar el proceso de forma reproducible, se construye un `DataFrame` con varios textos en español. Cada columna conserva una etapa diferente del procesamiento, de modo que el lector puede comparar el texto original con las representaciones transformadas y observar qué información se elimina o conserva.

In [ ]:
# %% Corpus tabular: conserva el texto original y muestra las transformaciones principales.
comentarios_ejemplo = [
    'Excelente explicación de inteligencia artificial, muy clara y útil.',
    'El código no funcionó en Google Colab y mostró un error de instalación.',
    '¿Qué librería de Python es mejor para analizar textos en español?',
    'El procesamiento del lenguaje natural ayuda a comprender documentos.',
    'La traducción automática permite comparar textos en distintos idiomas.'
]

def quitar_stopwords(texto):
    return ' '.join(palabra for palabra in texto.split() if palabra not in stop_words_es)

def aplicar_stemming(texto):
    return ' '.join(stemmer.stem(palabra) for palabra in texto.split())

df_texto = pd.DataFrame({'id_comentario': range(101, 106), 'texto_original': comentarios_ejemplo})
df_texto['texto_normalizado'] = df_texto['texto_original'].map(normalizar)
df_texto['texto_sin_stopwords'] = df_texto['texto_normalizado'].map(quitar_stopwords)
df_texto['texto_stemmed'] = df_texto['texto_sin_stopwords'].map(aplicar_stemming)
df_texto['num_palabras_originales'] = df_texto['texto_original'].str.split().str.len()
df_texto['num_palabras_filtradas'] = df_texto['texto_sin_stopwords'].str.split().str.len()

print('Corpus tabular de comentarios:')
display(df_texto)

frecuencia_corpus = Counter(' '.join(df_texto['texto_sin_stopwords']).split())
frecuencia_df = pd.DataFrame(
    frecuencia_corpus.most_common(10),
    columns=['termino', 'frecuencia']
)
print('Diez términos más frecuentes después del filtrado:')
display(frecuencia_df)

## Análisis lingüístico del corpus

Además del ejemplo individual, se analiza cada comentario del corpus con spaCy. La tabla resultante permite documentar POS, lematización y entidades nombradas para varios textos, como en un informe experimental.

In [ ]:
# %% Análisis lingüístico: resume POS, lemas, dependencias y entidades para cada comentario.
analisis_linguistico = []
for _, fila in df_texto.iterrows():
    documento = nlp(fila['texto_original'])
    analisis_linguistico.append({
        'id_comentario': fila['id_comentario'],
        'pos_principales': ', '.join(f'{token.text}:{token.pos_}' for token in documento if not token.is_punct),
        'lemas': ', '.join(token.lemma_ for token in documento if not token.is_punct),
        'dependencia_raiz': ', '.join(f'{token.text}->{token.head.text}' for token in documento if token.dep_ == 'ROOT'),
        'entidades': ', '.join(f'{ent.text} ({ent.label_})' for ent in documento.ents) or 'Ninguna'
    })

df_linguistico = pd.DataFrame(analisis_linguistico)
print('Análisis lingüístico por comentario:')
display(df_linguistico)

## 4. Term Frequency (TF)

La frecuencia de términos mide cuántas veces aparece cada palabra respecto al total de palabras del documento. Su valor está entre 0 y 1 cuando se usa frecuencia relativa. Los términos repetidos reciben mayor peso dentro del documento, aunque TF por sí sola no distingue palabras comunes en todo el corpus.

In [ ]:
# %% 4. TF: calcula manualmente la frecuencia relativa de cada término.
frecuencias = Counter(palabras)
total_terminos = len(palabras)
tf = {termino: frecuencia / total_terminos for termino, frecuencia in frecuencias.items()}
mostrar('4. Term Frequency:', tf)

## 5. Inverse Document Frequency (IDF)

La IDF reduce el peso de los términos que aparecen en muchos documentos y aumenta el de los términos más específicos. `TfidfVectorizer` calcula una versión suavizada de esta medida y permite observar la matriz TF-IDF resultante. En un corpus pequeño, las diferencias son orientativas; con más documentos, la discriminación temática es más representativa.

In [ ]:
# %% 5. IDF: obtiene el vocabulario y la IDF suavizada desde TfidfVectorizer.
vectorizador_tfidf = TfidfVectorizer()
matriz_tfidf = vectorizador_tfidf.fit_transform(corpus_ejemplo)
vocabulario = vectorizador_tfidf.get_feature_names_out()
idf = dict(zip(vocabulario, vectorizador_tfidf.idf_))

df_idf = pd.DataFrame({
    'termino': vocabulario,
    'idf': vectorizador_tfidf.idf_
}).sort_values('idf', ascending=False).reset_index(drop=True)
mostrar('5. IDF (términos más discriminativos):', df_idf.head(10))
print('Forma de la matriz TF-IDF:', matriz_tfidf.shape)

## 6. Part-of-speech Tagging

El etiquetado POS asigna una categoría gramatical a cada token, por ejemplo, sustantivo, verbo, adjetivo o preposición. Esta información permite estudiar la estructura del lenguaje y construir reglas lingüísticas. La salida debe leerse como pares token-categoría; la precisión depende del modelo de spaCy y del dominio del texto.

In [ ]:
# %% 6. POS tagging: identifica la categoría gramatical de cada token con spaCy.
doc = nlp('El presidente de Colombia anunció nuevas inversiones en tecnología en Bogotá.')
pos_tags = [(token.text, token.pos_, token.tag_) for token in doc]
mostrar('6. POS (token, POS, etiqueta):', pos_tags)

## 7. Lematización

La lematización transforma cada palabra a su lema o forma de diccionario, utilizando el contexto lingüístico del modelo. A diferencia del stemming, produce normalmente palabras válidas y facilita agrupar formas como verbos conjugados. El resultado puede variar si el modelo no reconoce correctamente el contexto o una palabra especializada.

In [ ]:
# %% 7. Lematización: devuelve la forma canónica de cada palabra mediante spaCy.
lemas = [(token.text, token.lemma_) for token in doc if not token.is_punct]
mostrar('7. Lemas:', lemas)

## 8. Dependency Parsing

El análisis de dependencias representa las relaciones sintácticas entre tokens y conecta cada palabra con su núcleo. Las parejas token-relación-núcleo permiten identificar sujetos, objetos, modificadores y el verbo principal. Esta representación es útil para análisis sintáctico, extracción de relaciones y comprensión estructural de una oración.

In [ ]:
# %% 8. Dependency parsing: muestra la relación sintáctica y el núcleo de cada token.
dependencias = [(token.text, token.dep_, token.head.text) for token in doc if not token.is_punct]
mostrar('8. Dependencias (token, relación, núcleo):', dependencias)

## 9. Named-Entity Recognition (NER)

El reconocimiento de entidades nombradas localiza expresiones que representan personas, lugares, organizaciones, fechas u otras clases. La salida incluye el texto de la entidad y la etiqueta asignada por spaCy. Estas predicciones sirven para extraer información, aunque deben revisarse en textos con nombres ambiguos o dominios especializados.

In [ ]:
# %% 9. NER: extrae entidades nombradas y su etiqueta semántica.
entidades = [(ent.text, ent.label_, ent.start_char, ent.end_char) for ent in doc.ents]
mostrar('9. Entidades (texto, etiqueta, inicio, fin):', entidades)

## 10. Generación de texto usando LLM

Una LLM predice una continuación probable a partir de un prompt. `max_new_tokens` limita la longitud generada, mientras que el muestreo y la temperatura introducen variación. La salida es texto generado, no una afirmación garantizada: debe evaluarse por coherencia, relevancia y posibles errores factuales.

In [ ]:
# %% 10. LLM: genera una continuación con una pipeline de Hugging Face.
generador_llm = pipeline(
    'text-generation',
    model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    device=DEVICE
)
prompt = 'La inteligencia artificial puede ayudar a la sociedad cuando'
resultado_llm = generador_llm(prompt, max_new_tokens=40, do_sample=True, temperature=0.7, return_full_text=False)
mostrar('10. Continuación generada:', resultado_llm[0]['generated_text'])

## 11. Question Answering

El sistema de Question Answering identifica dentro de un contexto el fragmento que mejor responde a una pregunta. En este laboratorio se utiliza un modelo entrenado para español y se muestran la pregunta y la respuesta extraída. La calidad depende de que la respuesta esté presente en el contexto y de que el modelo comprenda la formulación de la pregunta.

In [ ]:
# %% 11. Question Answering: responde directamente con un modelo QA en español.
modelo_qa = 'mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es'
tokenizador_qa = AutoTokenizer.from_pretrained(modelo_qa)
modelo_qa = AutoModelForQuestionAnswering.from_pretrained(modelo_qa)

contexto = 'El procesamiento del lenguaje natural es una rama de la inteligencia artificial que estudia el texto humano.'
pregunta = '¿Qué estudia el procesamiento del lenguaje natural?'
entradas_qa = tokenizador_qa(pregunta, contexto, return_tensors='pt', truncation=True)
with torch.no_grad():
    salidas_qa = modelo_qa(**entradas_qa)

inicio = int(torch.argmax(salidas_qa.start_logits))
fin = int(torch.argmax(salidas_qa.end_logits))
if fin < inicio:
    fin = inicio
ids_respuesta = entradas_qa['input_ids'][0][inicio:fin + 1]
respuesta_qa = tokenizador_qa.decode(ids_respuesta, skip_special_tokens=True)
mostrar('11. Respuesta:', {'pregunta': pregunta, 'respuesta': respuesta_qa})

## 12. Summarization

La generación de resúmenes condensa un texto conservando sus ideas centrales. El modelo seq2seq produce una versión más breve, pero puede omitir detalles o reformular información. Por ello se debe comparar el resumen con el texto original y comprobar que no introduzca datos que no estaban presentes.

In [ ]:
# %% 12. Summarization: genera un resumen en español directamente con un modelo seq2seq.
modelo_resumen = 'mrm8488/bert2bert_shared-spanish-finetuned-summarization'
tokenizador_resumen = AutoTokenizer.from_pretrained(modelo_resumen)
modelo_resumen = AutoModelForSeq2SeqLM.from_pretrained(modelo_resumen)

texto_largo = (
    'Las redes neuronales artificiales son modelos computacionales inspirados en el cerebro humano. '
    'Están formadas por nodos conectados en capas y aprenden patrones a partir de datos. '
    'Por ello se utilizan para clasificar información y realizar predicciones.'
)
entradas_resumen = tokenizador_resumen(
    texto_largo,
    return_tensors='pt',
    truncation=True,
    max_length=512
)
with torch.no_grad():
    ids_resumen = modelo_resumen.generate(
        **entradas_resumen,
        max_new_tokens=45,
        min_new_tokens=12,
        num_beams=4,
        early_stopping=True
    )
resumen = tokenizador_resumen.decode(ids_resumen[0], skip_special_tokens=True)
mostrar('12. Resumen:', resumen)

## 13. Sentence Similarity

La similitud de oraciones compara representaciones vectoriales de frases. La similitud coseno cercana a 1 indica direcciones semánticas parecidas, mientras que valores cercanos a 0 sugieren poca relación. El modelo multilingüe permite comparar frases en español, pero el resultado depende de la calidad del embedding y del contexto.

In [ ]:
# %% 13. Sentence similarity: compara embeddings multilingües mediante similitud coseno.
modelo_embeddings = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
oraciones = [
    'El coche avanza veloz por la autopista.',
    'El auto circula rápidamente por la carretera.'
]
embeddings = modelo_embeddings.encode(oraciones, normalize_embeddings=True)
similitud = cosine_similarity([embeddings[0]], [embeddings[1]])[0, 0]
mostrar('13. Similitud coseno:', round(float(similitud), 4))

## 14. Text Classification

La clasificación asigna una etiqueta a un texto a partir de ejemplos previamente etiquetados. TF-IDF convierte las opiniones en vectores y LogisticRegression aprende una frontera entre las clases Positivo y Negativo. La predicción es demostrativa porque el conjunto de entrenamiento es pequeño; un sistema real necesita más datos, validación y métricas como precisión, recall y F1.

In [ ]:
# %% 14. Clasificación: entrena una tubería TF-IDF + LogisticRegression para opiniones.
x_train = [
    'Excelente producto, llegó a tiempo y tiene gran calidad',
    'Pésimo servicio, el artículo llegó roto y tarde',
    'Me encantó la atención y la compra fue rápida',
    'Muy mala experiencia, no lo recomiendo para nada',
    'La calidad es fantástica y volvería a comprar',
    'El pedido fue un desastre y el soporte no ayudó'
]
y_train = ['Positivo', 'Negativo', 'Positivo', 'Negativo', 'Positivo', 'Negativo']
clasificador = LogisticRegression(max_iter=1000)
clasificador.fit(TfidfVectorizer().fit_transform(x_train), y_train)
vectorizador_clf = TfidfVectorizer()
x_train_vectorizado = vectorizador_clf.fit_transform(x_train)
clasificador.fit(x_train_vectorizado, y_train)
texto_prueba = 'El servicio al cliente fue horrible'
prediccion = clasificador.predict(vectorizador_clf.transform([texto_prueba]))[0]
mostrar('14. Predicción:', {'texto': texto_prueba, 'clase': prediccion})

## 15. Translation

La traducción automática transforma una frase del español al inglés mediante un modelo seq2seq entrenado con pares de idiomas. El resultado permite comprobar la transferencia de significado, pero puede perder matices, producir errores terminológicos o requerir revisión humana en contextos académicos y profesionales.

In [ ]:
# %% 15. Traducción: traduce directamente del español al inglés con un modelo seq2seq.
modelo_traduccion = 'Helsinki-NLP/opus-mt-es-en'
tokenizador_traduccion = AutoTokenizer.from_pretrained(modelo_traduccion)
modelo_traduccion = AutoModelForSeq2SeqLM.from_pretrained(modelo_traduccion)

texto_traducir = 'Este laboratorio de procesamiento de texto está completo.'
entradas_traduccion = tokenizador_traduccion(
    texto_traducir,
    return_tensors='pt',
    truncation=True
)
with torch.no_grad():
    ids_traduccion = modelo_traduccion.generate(
        **entradas_traduccion,
        max_new_tokens=80,
        num_beams=4
    )
traduccion = tokenizador_traduccion.decode(ids_traduccion[0], skip_special_tokens=True)
mostrar('15. Traducción ES -> EN:', traduccion)

## 16. Text Generation con n-gramas

El modelo de bigramas estima la siguiente palabra usando únicamente la palabra anterior y las frecuencias observadas en el corpus. Es transparente y fácil de explicar, pero genera texto limitado por el tamaño y el vocabulario de los datos. La salida permite comparar un método estadístico sencillo con la generación basada en una LLM.

In [ ]:
# %% 16. Generación n-gram: construye un modelo bigrama sencillo y decodifica texto.
tokens = normalizar(' '.join(corpus_ejemplo)).split()
bigramas = Counter(zip(tokens, tokens[1:]))

def generar_con_bigramas(inicio, longitud=12):
    salida = [inicio]
    for _ in range(longitud - 1):
        candidatos = {
            siguiente: frecuencia
            for (actual, siguiente), frecuencia in bigramas.items()
            if actual == salida[-1]
        }
        if not candidatos:
            break
        siguiente = max(candidatos, key=candidatos.get)
        salida.append(siguiente)
    return ' '.join(salida)

mostrar('16. Texto generado con bigramas:', generar_con_bigramas('el'))

## 17. Text mining

La minería de texto busca patrones útiles en una colección de documentos. Aquí se cuentan bigramas para detectar colocaciones frecuentes y se ordenan por aparición. En un proyecto real, este análisis puede ampliarse con n-gramas de distintos tamaños, medidas de asociación y visualizaciones para descubrir temas o relaciones relevantes.

In [ ]:
# %% 17. Text mining: extrae y ordena colocaciones de dos palabras por frecuencia.
ngrams_por_documento = []
for documento in corpus_ejemplo:
    tokens_documento = normalizar(documento).split()
    ngrams_por_documento.extend(zip(tokens_documento, tokens_documento[1:]))

frecuencia_ngrams = Counter(ngrams_por_documento)
df_ngrams = pd.DataFrame(
    [(' '.join(ngram), frecuencia) for ngram, frecuencia in frecuencia_ngrams.most_common(10)],
    columns=['bigram', 'frecuencia_absoluta']
)
mostrar('17. Bigramas más frecuentes:', df_ngrams)

## Resultados integrados del laboratorio

La siguiente tabla reúne indicadores de las distintas etapas. Esta vista facilita la comparación entre documentos y deja una evidencia compacta para el informe: reducción de palabras, tamaño del vocabulario TF-IDF, similitud semántica y predicción supervisada.

In [ ]:
# %% Resultados integrados: resume las métricas principales en tablas exportables al PDF.
resumen_resultados = pd.DataFrame({
    'indicador': [
        'Comentarios analizados',
        'Palabras originales promedio',
        'Palabras tras stopwords promedio',
        'Vocabulario TF-IDF',
        'Similitud coseno entre oraciones',
        'Predicción de clasificación'
    ],
    'valor': [
        len(df_texto),
        round(df_texto['num_palabras_originales'].mean(), 2),
        round(df_texto['num_palabras_filtradas'].mean(), 2),
        len(vocabulario),
        round(float(similitud), 4),
        prediccion
    ]
})

display(resumen_resultados)
print('Interpretación: el filtrado reduce el ruido léxico y TF-IDF conserva términos discriminativos;')
print('la similitud cuantifica cercanía semántica y el clasificador asigna una categoría al texto de prueba.')

# %% Conclusiones y análisis global
## Conclusiones y análisis global

El laboratorio muestra un flujo completo de procesamiento de texto: primero se normaliza el corpus y se reducen variaciones mediante stemming o lematización; después se eliminan términos poco informativos y se calculan pesos TF e IDF. Estas transformaciones convierten el lenguaje en representaciones que pueden utilizarse para clasificación, búsqueda y minería de patrones.

Los resultados lingüísticos de spaCy permiten observar categorías gramaticales, lemas, relaciones sintácticas y entidades nombradas. En conjunto, POS, lematización, parsing y NER aportan estructura semántica y sintáctica al texto, aunque su calidad depende del modelo `es_core_news_sm` y del dominio de los ejemplos.

Los modelos de Transformers amplían el laboratorio con generación, preguntas y respuestas, resumen y traducción. Son técnicas potentes, pero requieren modelos descargados, memoria suficiente y revisión de sus respuestas. La similitud semántica aporta una medida numérica para comparar frases, mientras que la clasificación TF-IDF + LogisticRegression demuestra un flujo supervisado reproducible.

Como limitaciones, el corpus y el conjunto de entrenamiento son pequeños y sirven para fines didácticos, no para evaluar rendimiento en producción. Para una evaluación formal se deberían añadir más documentos, separar entrenamiento y prueba, calcular métricas, comparar modelos y revisar manualmente las salidas generativas. El modelo de bigramas, aunque sencillo, ayuda a entender la base estadística de la generación de texto y proporciona una referencia interpretable frente a las LLM.